In [ ]:
# Install dependencies if needed
# %pip install -r ../requirements.txt
# dbutils.library.restartPython()

In [ ]:
import sys
import time
from pprint import pprint

sys.path.insert(0, '..')

from multiAgentSystem.agents.parser import parser_node
from experiments.mlflow_setup import (
    setup_experiment,
    create_agent_run,
    log_agent_metrics,
    log_state_snapshot,
    enable_autologging
)
from experiments.mock_data import PARSER_TEST_STATES, get_test_state

print("✓ Imports successful")

In [ ]:
# Setup MLflow experiment
enable_autologging()
experiment_id = setup_experiment("parser")
print(f"Experiment ID: {experiment_id}")

In [ ]:
# Configuration: Update this path to your actual Spark logs
# For Databricks, this should be a Unity Catalog Volume path
REAL_LOGS_PATH = "/Volumes/amruthcatalogtest/default/testsparklogs/sample/"

print(f"Using logs path: {REAL_LOGS_PATH}")
print("\n⚠️ If tests fail, ensure the logs path exists and contains Spark log files.")

In [ ]:
def run_parser_test(scenario_name: str, logs_path_override: str = None, verbose: bool = True):
    """
    Run a single parser test scenario with MLflow tracking.
    """
    scenario = get_test_state("parser", scenario_name)
    test_state = scenario["state"].copy()
    expected = scenario["expected"]
    
    # Override logs path if provided
    if logs_path_override and test_state.get("logs_path"):
        test_state["logs_path"] = logs_path_override
    
    with create_agent_run("parser", scenario=scenario_name) as run:
        log_state_snapshot(test_state, prefix="input")
        
        start_time = time.time()
        try:
            result = parser_node(test_state)
            success = True
        except Exception as e:
            result = {"error": str(e)}
            success = False
        latency_ms = (time.time() - start_time) * 1000
        
        # Calculate metrics
        evidence_map = result.get("evidence_map", {})
        total_patterns = len(evidence_map)
        total_occurrences = sum(e.get("count", 0) for e in evidence_map.values())
        has_error = "error" in str(result.get("evidence_map", {}).keys()).lower() or "error" in result
        
        log_agent_metrics(
            latency_ms=latency_ms,
            success=success,
            additional_metrics={
                "evidence_patterns": total_patterns,
                "evidence_occurrences": total_occurrences,
                "has_error_in_result": 1.0 if has_error else 0.0,
            }
        )
        
        log_state_snapshot(result, prefix="output")
        
        # Check expected outcomes
        passed = True
        if expected.get("has_evidence_map") and not evidence_map:
            passed = False
        if expected.get("has_error") and not has_error:
            passed = False
        if expected.get("has_summary") and not result.get("evidence_summary"):
            passed = False
        
        import mlflow
        mlflow.log_metric("test_passed", 1.0 if passed else 0.0)
        
        if verbose:
            status = "✅ PASSED" if passed else "❌ FAILED"
            print(f"\n{status} - {scenario_name}")
            print(f"  Description: {scenario['description']}")
            print(f"  Latency: {latency_ms:.2f}ms")
            print(f"  Evidence patterns: {total_patterns}")
            print(f"  Total occurrences: {total_occurrences}")
            print(f"  Has error: {has_error}")
            if result.get("evidence_summary"):
                print(f"  Summary preview: {result['evidence_summary'][:200]}...")
        
        return result, passed, latency_ms

## Test 1: Valid Search

Search real logs with valid keywords. Requires actual log files.

In [ ]:
# This test requires real log files
result_1, passed_1, latency_1 = run_parser_test("valid_search", logs_path_override=REAL_LOGS_PATH)

## Test 2: Error Handling - Missing Path

Empty logs path should return an error in evidence map.

In [ ]:
result_2, passed_2, latency_2 = run_parser_test("no_path_error")

## Test 3: Error Handling - Missing Keywords

Empty keywords should return an error in evidence map.

In [ ]:
result_3, passed_3, latency_3 = run_parser_test("no_keywords_error")

## Test 4: GC Analysis Trigger

GC-related keywords should trigger GC log analysis. Requires actual log files with GC data.

In [ ]:
result_4, passed_4, latency_4 = run_parser_test("gc_analysis_trigger", logs_path_override=REAL_LOGS_PATH)

## Summary

In [ ]:
print("=" * 60)
print("PARSER AGENT TEST SUMMARY")
print("=" * 60)

tests = [
    ("valid_search", passed_1, latency_1),
    ("no_path_error", passed_2, latency_2),
    ("no_keywords_error", passed_3, latency_3),
    ("gc_analysis_trigger", passed_4, latency_4),
]

total_passed = sum(1 for _, passed, _ in tests if passed)
avg_latency = sum(lat for _, _, lat in tests) / len(tests)

for name, passed, latency in tests:
    status = "✅" if passed else "❌"
    print(f"  {status} {name}: {latency:.2f}ms")

print("=" * 60)
print(f"Total: {total_passed}/{len(tests)} passed")
print(f"Average latency: {avg_latency:.2f}ms")
print("=" * 60)